# HW5 — Colab Training Runner

Одноразовый раннер для бесплатной Colab T4. Делает полный цикл HW5: подготовка sample из Lenta.ru, baseline-оценка, QLoRA-дообучение `Qwen/Qwen2.5-1.5B` с `torch.profiler`, повторная оценка, упаковка `dz5/artifacts_hw5/` в zip и скачивание.

Отчётный ноутбук `hw5.ipynb` лежит рядом и читает уже зафиксированные артефакты — сюда смотреть не нужно.

**Перед запуском:** Runtime → Change runtime type → T4 GPU. Если повторно запускаешь после частично неудачного прогона — шаги, чьи артефакты уже на диске, автоматически пропустятся.

In [ ]:
!git clone --depth 1 -b hw_5 https://github.com/dmagog/aith_DL_NLP.git
%cd aith_DL_NLP/dz5

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '—')
!nvidia-smi

In [ ]:
!wget -q -nc https://github.com/yutkin/Lenta.Ru-News-Dataset/releases/download/v1.1/lenta-ru-news.csv.bz2
!ls -lh lenta-ru-news.csv.bz2

In [ ]:
import subprocess
from pathlib import Path

cmd = [
    'python', '-m', 'src.run_full',
    '--corpus-path', 'lenta-ru-news.csv.bz2',
    '--out', 'artifacts_hw5',
    '--sample-size', '12000',
    '--eval-size', '500',
    '--num-train-epochs', '1',
    '--harness-limit', '200',
    '--perplexity-samples', '200',
    '--seed', '42',
]

if Path('artifacts_hw5/metrics_before.json').exists():
    cmd.append('--skip-baseline')
    print('baseline уже есть на диске — пропускаем его пересчёт')

if Path('artifacts_hw5/lora_adapter/adapter_config.json').exists():
    cmd.append('--skip-train')
    print('LoRA-адаптер уже на диске — пропускаем train')

print('running:', ' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
from pathlib import Path

required = [
    'metrics_before.json',
    'metrics_after.json',
    'harness_before.json',
    'harness_after.json',
    'basket_before.md',
    'basket_after.md',
    'train_metrics.json',
    'profiler_summary.md',
    'lora_adapter/adapter_config.json',
    'lora_adapter/adapter_model.safetensors',
]
missing = [r for r in required if not Path('artifacts_hw5', r).exists()]
if missing:
    raise RuntimeError(f'Прогон не завершён, на диске нет: {missing}')

import shutil
shutil.make_archive('hw5_artifacts', 'zip', 'artifacts_hw5')
from google.colab import files
files.download('hw5_artifacts.zip')